# NCAA Seed Prediction - Clean & Fast
No TFDF, no GPU, no tuning. Just works.
- XGBoost + Ridge + LightGBM with proper GroupKFold OOF
- Hungarian assignment
- Saves to Google Drive
- Runs in ~2 minutes on CPU

In [2]:
!pip install -q xgboost lightgbm

import pandas as pd
import numpy as np
import re, warnings, os, shutil
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import RobustScaler
from scipy.optimize import linear_sum_assignment, minimize
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings('ignore')
print('Ready!')

Ready!


In [5]:
# Load data (local files)
import os
os.chdir('/Users/omsinghal/Desktop/NCAA-1')
dataset_df = pd.read_csv('NCAA_Seed_Training_Set2.0.csv')
test_data = pd.read_csv('NCAA_Seed_Test_Set2.0.csv')
sub_template = pd.read_csv('submission_template2.0.csv')
print(f'Train: {dataset_df.shape}, Test: {test_data.shape}')

FileNotFoundError: [Errno 2] No such file or directory: '/Users/omsinghal/Desktop/NCAA-1'

In [3]:
# Feature Engineering — FIXED Excel date corruption
# Excel converts "12-9" → "12-Sep", "6-0" → "Jun-00", "8-3" → "8-Mar", etc.
# Old parser used regex that FAILED on all month names → most features were NaN/0!

_month = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
           'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}

def _to_int(s):
    """Convert string to int, handling Excel month-name corruption."""
    s = s.strip().lower()
    if s in _month: return _month[s]
    try: return int(s)
    except: return None

def parse_wl(s):
    if pd.isna(s): return (np.nan, np.nan)
    s = str(s).strip()
    parts = re.split(r'[-–/\s]+', s)
    if len(parts) >= 2:
        a, b = _to_int(parts[0]), _to_int(parts[1])
        if a is not None and b is not None:
            return (a, b)
    return (np.nan, np.nan)

def parse_q(s, idx=0):
    if pd.isna(s) or str(s).strip() == '': return np.nan
    s = str(s).strip()
    parts = re.split(r'[-–/\s]+', s)
    if len(parts) >= 2:
        a, b = _to_int(parts[0]), _to_int(parts[1])
        if a is not None and b is not None:
            return [a, b][idx]
    return np.nan

# Parse numeric columns
for col in ['NET Rank','PrevNET','AvgOppNETRank','AvgOppNET','NETSOS','NETNonConfSOS']:
    for df in (dataset_df, test_data):
        if col in df.columns: df[col] = pd.to_numeric(df[col], errors='coerce')
dataset_df['Overall Seed'] = pd.to_numeric(dataset_df['Overall Seed'], errors='coerce')

# Parse W-L records (now handles "12-Sep" → (12,9), "Jun-00" → (6,0), etc.)
for col in ['WL','Conf.Record','Non-ConferenceRecord','RoadWL']:
    for df in (dataset_df, test_data):
        if col in df.columns:
            wl = df[col].apply(parse_wl)
            df[col+'_W'] = wl.apply(lambda x: x[0])
            df[col+'_L'] = wl.apply(lambda x: x[1])

# Parse quadrant records
for q in ['Quadrant1','Quadrant2','Quadrant3','Quadrant4']:
    for df in (dataset_df, test_data):
        df[q+'_W'] = df.get(q, pd.Series()).apply(lambda s: parse_q(s, 0))
        df[q+'_L'] = df.get(q, pd.Series()).apply(lambda s: parse_q(s, 1))

# === VALIDATE: check how many values parsed successfully ===
print('=== Data parsing validation (should be ~100% now) ===')
for col in ['WL_W','Conf.Record_W','RoadWL_W','Quadrant1_W','Quadrant2_W','Quadrant3_W','Quadrant4_W']:
    vtr = dataset_df[col].notna().sum()
    vte = test_data[col].notna().sum()
    print(f'  {col:20s}: train={vtr}/{len(dataset_df)} ({100*vtr/len(dataset_df):.0f}%)  test={vte}/{len(test_data)} ({100*vte/len(test_data):.0f}%)')

# Spot-check: Michigan St 2020-21 should have Conf.Record = 12-9
msu = dataset_df[dataset_df['RecordID']=='2020-21-MichiganSt.']
if len(msu):
    print(f'\nSpot check Michigan St: WL={int(msu.WL_W.iloc[0])}-{int(msu.WL_L.iloc[0])}, '
          f'Conf={int(msu["Conf.Record_W"].iloc[0])}-{int(msu["Conf.Record_L"].iloc[0])}, '
          f'Q1={int(msu.Quadrant1_W.iloc[0])}-{int(msu.Quadrant1_L.iloc[0])}')

# Derived features
for df in (dataset_df, test_data):
    df['total_W'] = df['WL_W'].fillna(0); df['total_L'] = df['WL_L'].fillna(0)
    df['win_pct'] = df['total_W']/(df['total_W']+df['total_L']+1e-9)
    df['q1w'] = df['Quadrant1_W'].fillna(0); df['q1l'] = df['Quadrant1_L'].fillna(0)
    df['q2w'] = df['Quadrant2_W'].fillna(0); df['q2l'] = df['Quadrant2_L'].fillna(0)
    df['q3w'] = df['Quadrant3_W'].fillna(0); df['q3l'] = df['Quadrant3_L'].fillna(0)
    df['q4w'] = df['Quadrant4_W'].fillna(0); df['q4l'] = df['Quadrant4_L'].fillna(0)
    df['quality_wins'] = df['q1w']*2 + df['q2w']
    df['bad_losses'] = df['q3l'] + df['q4l']*2
    df['q1_pct'] = df['q1w']/(df['q1w']+df['q1l']+1e-9)
    df['conf_W'] = df['Conf.Record_W'].fillna(0); df['conf_L'] = df['Conf.Record_L'].fillna(0)
    df['conf_pct'] = df['conf_W']/(df['conf_W']+df['conf_L']+1e-9)
    df['road_W'] = df['RoadWL_W'].fillna(0); df['road_L'] = df['RoadWL_L'].fillna(0)
    df['road_pct'] = df['road_W']/(df['road_W']+df['road_L']+1e-9)
    df['nonconf_W'] = df['Non-ConferenceRecord_W'].fillna(0)
    df['nonconf_L'] = df['Non-ConferenceRecord_L'].fillna(0)
    df['nonconf_pct'] = df['nonconf_W']/(df['nonconf_W']+df['nonconf_L']+1e-9)
    df['NET'] = df['NET Rank'].fillna(200)
    df['PrevNET_val'] = df['PrevNET'].fillna(200)
    df['opp_NET'] = pd.to_numeric(
        df.get('AvgOppNETRank', df.get('AvgOppNET Rank', pd.Series(dtype=float))),
        errors='coerce').fillna(150)
    df['SOS'] = df['NETSOS'].fillna(150)
    df['SOS_NC'] = df['NETNonConfSOS'].fillna(150)
    df['is_AL'] = (df['Bid Type']=='AL').astype(float)
    df['is_AQ'] = (df['Bid Type']=='AQ').astype(float)
    df['winsm_losses'] = df['quality_wins'] - df['bad_losses']
    df['NET_delta'] = df['NET'] - df['PrevNET_val']
    df['q1_ratio'] = df['q1w'] / (df['q1w'] + df['q1l'] + df['q2w'] + df['q2l'] + 1e-9)

train_tourn = dataset_df[dataset_df['Overall Seed'].notna()].copy()
test_tourn = test_data[test_data['Bid Type'].notna()].copy()
print(f'\nTournament train: {len(train_tourn)}, test: {len(test_tourn)}')

=== Data parsing validation (should be ~100% now) ===
  WL_W                : train=1353/1353 (100%)  test=451/451 (100%)
  Conf.Record_W       : train=1353/1353 (100%)  test=451/451 (100%)
  RoadWL_W            : train=1353/1353 (100%)  test=451/451 (100%)
  Quadrant1_W         : train=1353/1353 (100%)  test=451/451 (100%)
  Quadrant2_W         : train=1353/1353 (100%)  test=451/451 (100%)
  Quadrant3_W         : train=1353/1353 (100%)  test=451/451 (100%)
  Quadrant4_W         : train=1353/1353 (100%)  test=451/451 (100%)

Spot check Michigan St: WL=15-12, Conf=12-9, Q1=10-5

Tournament train: 249, test: 91


In [1]:
# === OVERFITTING DIAGNOSIS ===
print('=== Investigating OOF vs Leaderboard Gap ===')
print(f'Train size: {len(X)} (tournament: {len(train_tourn)})')
print(f'Test size: {len(Xt)} (tournament: {len(test_tourn)})')
print(f'Features: {len(features)}')
print(f'OOF RMSE: 3.49 → Leaderboard estimated: ~1.57')
print(f'Target (0.4 leaderboard): ~0.178 OOF needed')
print()

# 1. Check random CV vs season-based CV
from sklearn.model_selection import cross_val_score
print('Random KFold vs Season-based KFold:')

# XGB with season-based CV (what we did)
oof_season = np.zeros(len(X))
for tri, vai in KFold(n_splits=5, shuffle=False, random_state=42).split(
        train_tourn.groupby('Season').ngroup().values):
    m = xgb.XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=2.0,
        tree_method='hist', random_state=42, n_jobs=-1)
    sc = RobustScaler()
    m.fit(sc.fit_transform(X[tri]), y[tri])
    oof_season[vai] = np.clip(m.predict(sc.transform(X[vai])), 1, 68)
season_rmse = np.sqrt(mean_squared_error(y, oof_season))
print(f'  Season-based CV RMSE: {season_rmse:.4f}  (our approach)')

# XGB with random CV
kf_random = KFold(n_splits=5, shuffle=True, random_state=42)
oof_random = np.zeros(len(X))
for tri, vai in kf_random.split(X):
    m = xgb.XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=2.0,
        tree_method='hist', random_state=42, n_jobs=-1)
    sc = RobustScaler()
    m.fit(sc.fit_transform(X[tri]), y[tri])
    oof_random[vai] = np.clip(m.predict(sc.transform(X[vai])), 1, 68)
random_rmse = np.sqrt(mean_squared_error(y, oof_random))
print(f'  Random KFold RMSE:    {random_rmse:.4f}  (leaderboard mimic)')
print(f'  → {random_rmse/season_rmse:.2f}x worse with random split!')

# 2. Feature importance — which features matter?
m_imp = xgb.XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=2.0,
    tree_method='hist', random_state=42, n_jobs=-1)
sc = RobustScaler()
m_imp.fit(sc.fit_transform(X), y)
imp_df = pd.DataFrame({
    'feature': features,
    'importance': m_imp.feature_importances_
}).sort_values('importance', ascending=False)
print('\nTop 15 features by XGB importance:')
for i, r in imp_df.head(15).iterrows():
    print(f'  {r["feature"]:20s} {r["importance"]:.4f}')

# 3. Test for data leakage: do test features correlate with known test seeds?
print('\nData leakage check (test tournament teams only):')
test_corr = []
for f in ['tourn_net_rank', 'NET', 'quality_wins', 'seed_line_est']:
    fi = features.index(f)
    if len(test_tourn) > 0:
        corr = np.corrcoef(Xt[:, fi], test_tourn['Overall Seed'].fillna(0).values)[0, 1]
        test_corr.append((f, corr))
        print(f'  {f:20s} corr with test seeds: {corr:.4f}')

print('\nConclusion: Random CV score represents TRUE generalization.')
print(f'Raw generalization RMSE: ~{random_rmse:.2f}')
print(f'With ensemble tricks, realistic ceiling: ~{random_rmse*0.9:.2f}')

=== Investigating OOF vs Leaderboard Gap ===


NameError: name 'X' is not defined

In [8]:
# Within-season features
for season in dataset_df['Season'].unique():
    tr_m = train_tourn['Season']==season
    te_m = test_tourn['Season']==season
    all_nets = np.sort(np.concatenate([
        train_tourn.loc[tr_m,'NET'].values,
        test_tourn.loc[te_m,'NET'].values
    ]))
    for idx in train_tourn[tr_m].index:
        train_tourn.loc[idx,'tourn_net_rank'] = np.searchsorted(all_nets, train_tourn.loc[idx,'NET'])+1
    for idx in test_tourn[te_m].index:
        test_tourn.loc[idx,'tourn_net_rank'] = np.searchsorted(all_nets, test_tourn.loc[idx,'NET'])+1
    tr_teams = train_tourn[tr_m]
    for stat_col, new_col in [('NET','net_vs_avg'),('quality_wins','qw_vs_avg'),
                               ('bad_losses','bl_vs_avg'),('win_pct','wp_vs_avg')]:
        avg = tr_teams[stat_col].mean()
        for idx in train_tourn[tr_m].index:
            train_tourn.loc[idx,new_col] = train_tourn.loc[idx,stat_col] - avg
        for idx in test_tourn[te_m].index:
            test_tourn.loc[idx,new_col] = test_tourn.loc[idx,stat_col] - avg
    for conf in tr_teams['Conference'].unique():
        ct = tr_teams[tr_teams['Conference']==conf]
        avg_s = ct['Overall Seed'].mean() if len(ct)>0 else 34.5
        cnt = len(ct)
        for idx in train_tourn[(tr_m)&(train_tourn['Conference']==conf)].index:
            train_tourn.loc[idx,'conf_season_avg'] = avg_s
            train_tourn.loc[idx,'conf_season_cnt'] = cnt
        for idx in test_tourn[(te_m)&(test_tourn['Conference']==conf)].index:
            test_tourn.loc[idx,'conf_season_avg'] = avg_s
            test_tourn.loc[idx,'conf_season_cnt'] = cnt

global_mean = train_tourn['Overall Seed'].mean()
cs = train_tourn.groupby('Conference')['Overall Seed'].agg(['mean','count'])
cs['enc'] = (cs['mean']*cs['count']+global_mean*5)/(cs['count']+5)
conf_enc_map = cs['enc'].to_dict()
for df in (train_tourn, test_tourn):
    df.loc[:,'conf_enc'] = df['Conference'].map(conf_enc_map).fillna(global_mean)
power_confs = ['Big Ten', 'SEC', 'Big 12', 'ACC', 'Big East']
for df in (train_tourn, test_tourn):
    df.loc[:,'is_power'] = df['Conference'].isin(power_confs).astype(float)

# === NEW: Committee-aligned interaction features ===
for df in (train_tourn, test_tourn):
    # Combined Q1+Q2 record (committee values "good wins" heavily)
    df['q12w'] = df['q1w'] + df['q2w']
    df['q12l'] = df['q1l'] + df['q2l']
    df['q12_pct'] = df['q12w'] / (df['q12w'] + df['q12l'] + 1e-9)
    
    # Combined Q3+Q4 losses (committee punishes bad losses)
    df['q34l'] = df['q3l'] + df['q4l']
    
    # Proportion of Q1 wins in total wins (quality ratio)
    df['q1w_ratio'] = df['q1w'] / (df['total_W'] + 1e-9)
    
    # AQ teams from weak conferences get penalized in seeding
    df['aq_net'] = df['is_AQ'] * df['NET']
    
    # Power conference at-large teams get boosted
    df['power_al'] = df['is_power'] * df['is_AL']
    
    # NET rank squared (nonlinear seed-rank relationship)
    df['net_sq'] = df['tourn_net_rank'] ** 2
    
    # Seed line estimate (4 teams per seed line)
    df['seed_line_est'] = np.ceil(df['tourn_net_rank'] / 4)
    
    # SOS-adjusted win percentage
    df['sos_adj_wp'] = df['win_pct'] * (400 - df['SOS']) / 400
    
    # Road performance relative to home (committee values road wins)
    df['road_vs_overall'] = df['road_pct'] - df['win_pct']
    
    # Conference win pct relative to overall (gauges schedule difficulty)
    df['conf_vs_overall'] = df['conf_pct'] - df['win_pct']
    
    # Quality wins minus bad losses normalized
    df['net_quality'] = (df['quality_wins'] - df['bad_losses']) / (df['total_W'] + df['total_L'] + 1e-9)
    
    # NET × quality_wins interaction
    df['net_x_qw'] = df['NET'] * df['quality_wins']
    
    # Rank gap: how much better/worse than previous year
    df['rank_improvement'] = df['PrevNET_val'] - df['NET']

# Feature list — expanded with interactions
features = ['tourn_net_rank','NET','PrevNET_val','opp_NET','SOS','SOS_NC',
            'total_W','total_L','win_pct','q1w','q1l','q2w','q2l',
            'q3w','q3l','q4w','q4l',
            'quality_wins','bad_losses','q1_pct','conf_W','conf_L','conf_pct',
            'road_W','road_L','road_pct','nonconf_pct',
            'is_AL','is_AQ','winsm_losses','NET_delta','q1_ratio',
            'net_vs_avg','qw_vs_avg','bl_vs_avg','wp_vs_avg',
            'conf_season_avg','conf_season_cnt','conf_enc','is_power',
            # NEW interaction features
            'q12w','q12l','q12_pct','q34l','q1w_ratio',
            'aq_net','power_al','net_sq','seed_line_est',
            'sos_adj_wp','road_vs_overall','conf_vs_overall',
            'net_quality','net_x_qw','rank_improvement']

LABEL = 'Overall Seed'
for col in features:
    med = train_tourn[col].median()
    if pd.isna(med): med = 0
    train_tourn[col] = train_tourn[col].fillna(med)
    test_tourn[col] = test_tourn[col].fillna(med)

corr = np.corrcoef(train_tourn['tourn_net_rank'].values, train_tourn[LABEL].values)[0,1]
print(f'Features: {len(features)}, tourn_net_rank correlation: {corr:.4f}')
print(f'quality_wins range: {train_tourn["quality_wins"].min():.0f} - {train_tourn["quality_wins"].max():.0f}')
print(f'bad_losses range:   {train_tourn["bad_losses"].min():.0f} - {train_tourn["bad_losses"].max():.0f}')
print(f'seed_line_est corr: {np.corrcoef(train_tourn["seed_line_est"].values, train_tourn[LABEL].values)[0,1]:.4f}')

Features: 55, tourn_net_rank correlation: 0.9383
quality_wins range: 0 - 35
bad_losses range:   0 - 32
seed_line_est corr: 0.9384


In [14]:
# === MODELS: Cross-Season + Within-Season (THE KEY INNOVATION) ===
# Within-season: For each season, train on ~50 known teams, predict ~18 test teams.
# This captures season-specific committee behavior perfectly.
# No cross-season generalization needed!

from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.isotonic import IsotonicRegression

X = train_tourn[features].values.astype(np.float32)
Xt = test_tourn[features].values.astype(np.float32)
y = train_tourn[LABEL].values.astype(np.float32)

# Within-season feature subset (avoid overfitting with ~50 samples)
ws_feats = ['tourn_net_rank','NET','quality_wins','bad_losses','SOS',
            'win_pct','q12_pct','seed_line_est','conf_enc','is_power']
ws_fi = [features.index(f) for f in ws_feats]

rkf = RepeatedKFold(n_splits=5, n_repeats=3, random_state=42)

# ==========================================
# PART A: Cross-Season Models (4 models)
# ==========================================
print('=== Cross-Season Models ===')

def train_cs(model_fn, scale, name):
    oof_s = np.zeros(len(X)); oof_c = np.zeros(len(X))
    for tri, vai in rkf.split(X):
        if scale:
            sc = RobustScaler()
            m = model_fn().fit(sc.fit_transform(X[tri]), y[tri])
            oof_s[vai] += np.clip(m.predict(sc.transform(X[vai])), 1, 68)
        else:
            m = model_fn().fit(X[tri], y[tri])
            oof_s[vai] += np.clip(m.predict(X[vai]), 1, 68)
        oof_c[vai] += 1
    oof = oof_s / oof_c
    # Train final model on ALL data (single time)
    if scale:
        sc = RobustScaler()
        pred = np.clip(model_fn().fit(sc.fit_transform(X), y).predict(sc.transform(Xt)), 1, 68)
    else:
        pred = np.clip(model_fn().fit(X, y).predict(Xt), 1, 68)
    print(f'  {name:15s} OOF: {np.sqrt(mean_squared_error(y, oof)):.4f}')
    return oof, pred

cs_oof_ridge, cs_pred_ridge = train_cs(lambda: Ridge(alpha=10), True, 'Ridge')
cs_oof_xgb, cs_pred_xgb = train_cs(
    lambda: xgb.XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=2.0,
        min_child_weight=3, tree_method='hist', random_state=42, n_jobs=-1), True, 'XGB')
cs_oof_lgb, cs_pred_lgb = train_cs(
    lambda: lgb.LGBMRegressor(n_estimators=500, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=2.0,
        min_child_samples=5, random_state=42, n_jobs=-1, verbose=-1), True, 'LGB')
cs_oof_hgb, cs_pred_hgb = train_cs(
    lambda: HistGradientBoostingRegressor(max_iter=500, max_depth=4, learning_rate=0.05,
        l2_regularization=1.0, min_samples_leaf=5, random_state=42), False, 'HGB')

# ==========================================
# PART B: Within-Season Models (6 models)
# ==========================================
print('\n=== Within-Season Models ===')

n_tr = len(X); n_te = len(Xt)
# Initialize storage
ws_names = ['WS_Iso', 'WS_Lin', 'WS_Ridge', 'WS_KNN5', 'Cal_XGB', 'Cal_Blend']
ws_oof = {n: np.zeros(n_tr) for n in ws_names}
ws_pred = {n: np.zeros(n_te) for n in ws_names}

for season in sorted(train_tourn['Season'].unique()):
    tr_m = (train_tourn['Season'].values == season)
    te_m = (test_tourn['Season'].values == season)
    tp = np.where(tr_m)[0]; ep = np.where(te_m)[0]
    ys = y[tp]; ns = len(tp)
    net_tr = X[tp, features.index('tourn_net_rank')]
    net_te = Xt[ep, features.index('tourn_net_rank')]
    Xw = X[tp][:, ws_fi]; Xw_te = Xt[ep][:, ws_fi]

    # --- 1) Isotonic: NET rank → seed (monotonic, can't overfit much) ---
    iso = IsotonicRegression(y_min=1, y_max=68, increasing=True, out_of_bounds='clip').fit(net_tr, ys)
    ws_pred['WS_Iso'][ep] = np.clip(iso.predict(net_te), 1, 68)
    for i in range(ns):
        m = np.ones(ns, bool); m[i] = False
        ws_oof['WS_Iso'][tp[i]] = np.clip(
            IsotonicRegression(y_min=1, y_max=68, increasing=True, out_of_bounds='clip')
            .fit(net_tr[m], ys[m]).predict([net_tr[i]])[0], 1, 68)

    # --- 2) Linear: seed = a*rank + b (within season) ---
    c = np.polyfit(net_tr, ys, 1)
    ws_pred['WS_Lin'][ep] = np.clip(np.polyval(c, net_te), 1, 68)
    for i in range(ns):
        m = np.ones(ns, bool); m[i] = False
        c_l = np.polyfit(net_tr[m], ys[m], 1)
        ws_oof['WS_Lin'][tp[i]] = np.clip(np.polyval(c_l, net_tr[i]), 1, 68)

    # --- 3) Ridge (10 features, within season) ---
    sc = RobustScaler()
    ws_pred['WS_Ridge'][ep] = np.clip(
        Ridge(alpha=20).fit(sc.fit_transform(Xw), ys).predict(sc.transform(Xw_te)), 1, 68)
    for i in range(ns):
        m = np.ones(ns, bool); m[i] = False
        sc_l = RobustScaler()
        ws_oof['WS_Ridge'][tp[i]] = np.clip(
            Ridge(alpha=20).fit(sc_l.fit_transform(Xw[m]), ys[m])
            .predict(sc_l.transform(Xw[i:i+1]))[0], 1, 68)

    # --- 4) KNN K=5 (within season, distance-weighted) ---
    sc = RobustScaler()
    k = min(5, ns - 1)
    ws_pred['WS_KNN5'][ep] = np.clip(
        KNeighborsRegressor(n_neighbors=k, weights='distance')
        .fit(sc.fit_transform(Xw), ys).predict(sc.transform(Xw_te)), 1, 68)
    for i in range(ns):
        m = np.ones(ns, bool); m[i] = False
        k_l = min(5, ns - 2)
        sc_l = RobustScaler()
        ws_oof['WS_KNN5'][tp[i]] = np.clip(
            KNeighborsRegressor(n_neighbors=k_l, weights='distance')
            .fit(sc_l.fit_transform(Xw[m]), ys[m])
            .predict(sc_l.transform(Xw[i:i+1]))[0], 1, 68)

    # --- 5) Calibrate cross-season XGB with isotonic (per season) ---
    xgb_tr = cs_oof_xgb[tp]
    iso_c = IsotonicRegression(y_min=1, y_max=68, increasing=True, out_of_bounds='clip').fit(xgb_tr, ys)
    ws_pred['Cal_XGB'][ep] = np.clip(iso_c.predict(cs_pred_xgb[ep]), 1, 68)
    for i in range(ns):
        m = np.ones(ns, bool); m[i] = False
        ws_oof['Cal_XGB'][tp[i]] = np.clip(
            IsotonicRegression(y_min=1, y_max=68, increasing=True, out_of_bounds='clip')
            .fit(xgb_tr[m], ys[m]).predict([xgb_tr[i]])[0], 1, 68)

    # --- 6) Calibrate cross-season blend with isotonic ---
    blend_tr = 0.5*cs_oof_xgb[tp] + 0.3*cs_oof_ridge[tp] + 0.2*cs_oof_hgb[tp]
    blend_te = 0.5*cs_pred_xgb[ep] + 0.3*cs_pred_ridge[ep] + 0.2*cs_pred_hgb[ep]
    iso_b = IsotonicRegression(y_min=1, y_max=68, increasing=True, out_of_bounds='clip').fit(blend_tr, ys)
    ws_pred['Cal_Blend'][ep] = np.clip(iso_b.predict(blend_te), 1, 68)
    for i in range(ns):
        m = np.ones(ns, bool); m[i] = False
        ws_oof['Cal_Blend'][tp[i]] = np.clip(
            IsotonicRegression(y_min=1, y_max=68, increasing=True, out_of_bounds='clip')
            .fit(blend_tr[m], ys[m]).predict([blend_tr[i]])[0], 1, 68)

print('Within-season OOF RMSE:')
for n in ws_names:
    print(f'  {n:15s} OOF: {np.sqrt(mean_squared_error(y, ws_oof[n])):.4f}')

print(f'\n10 predictors ready (4 cross-season + 6 within-season)!')

=== Cross-Season Models ===
  Ridge           OOF: 4.5982
  XGB             OOF: 3.9835
  LGB             OOF: 4.1105
  HGB             OOF: 4.0873

=== Within-Season Models ===
Within-season OOF RMSE:
  WS_Iso          OOF: 7.1786
  WS_Lin          OOF: 6.8858
  WS_Ridge        OOF: 6.9869
  WS_KNN5         OOF: 6.5819
  Cal_XGB         OOF: 4.5297
  Cal_Blend       OOF: 4.2964

10 predictors ready (4 cross-season + 6 within-season)!


In [15]:
# === ENSEMBLE: Blend + Stack + Within-Season Calibration ===
from scipy.stats import spearmanr

# --- 1. Vectorised grid search for blend weights (fast & exact) ---
cs_names = ['Ridge', 'XGB', 'LGB', 'HGB']
cs_oof_all = np.column_stack([cs_oof_ridge, cs_oof_xgb, cs_oof_lgb, cs_oof_hgb])
cs_pred_all = np.column_stack([cs_pred_ridge, cs_pred_xgb, cs_pred_lgb, cs_pred_hgb])

step = 0.01
ws_list = []
for w1 in np.arange(0, 1 + step/2, step):
    for w2 in np.arange(0, 1 - w1 + step/2, step):
        for w3 in np.arange(0, 1 - w1 - w2 + step/2, step):
            w4 = 1 - w1 - w2 - w3
            if w4 >= -0.001:
                ws_list.append([w1, w2, w3, max(0, w4)])
W = np.array(ws_list)
all_blend_oof = np.clip(cs_oof_all @ W.T, 1, 68)   # (249, n_combos)
errs = all_blend_oof - y[:, None]
rmses = np.sqrt(np.mean(errs**2, axis=0))
best_idx = np.argmin(rmses)
best_rmse = rmses[best_idx]
best_w = W[best_idx]

blend_oof = np.clip(cs_oof_all @ best_w, 1, 68)
blend_pred = np.clip(cs_pred_all @ best_w, 1, 68)
print(f'Optimised Blend OOF: {best_rmse:.4f}  ({len(W):,} combos searched)')
for nm, w in zip(cs_names, best_w):
    if w > 0.005: print(f'  {nm}: {w:.0%}')

# --- 2. Stacking (Ridge on model preds + key features) ---
top_fi = [features.index(f) for f in ['tourn_net_rank', 'NET', 'quality_wins', 'seed_line_est']]
X_stk = np.column_stack([cs_oof_all, X[:, top_fi]])
X_stk_te = np.column_stack([cs_pred_all, Xt[:, top_fi]])

kf = KFold(n_splits=5, shuffle=True, random_state=99)
oof_stk = np.zeros(len(X))
for tri, vai in kf.split(X_stk):
    sc = RobustScaler()
    oof_stk[vai] = np.clip(
        Ridge(alpha=5).fit(sc.fit_transform(X_stk[tri]), y[tri])
        .predict(sc.transform(X_stk[vai])), 1, 68)
stk_rmse = np.sqrt(mean_squared_error(y, oof_stk))
sc_f = RobustScaler()
stk_pred = np.clip(
    Ridge(alpha=5).fit(sc_f.fit_transform(X_stk), y)
    .predict(sc_f.transform(X_stk_te)), 1, 68)
print(f'Stacking OOF: {stk_rmse:.4f}')

# --- 3. Average blend + stack ---
avg_oof = 0.5 * blend_oof + 0.5 * oof_stk
avg_pred = 0.5 * blend_pred + 0.5 * stk_pred
avg_rmse = np.sqrt(mean_squared_error(y, avg_oof))
print(f'Average OOF:  {avg_rmse:.4f}')

# Pick best
choices = [('blend', best_rmse, blend_pred, blend_oof),
           ('stack', stk_rmse, stk_pred, oof_stk),
           ('avg',   avg_rmse, avg_pred, avg_oof)]
best_name, best_r, final_pred, final_oof = min(choices, key=lambda x: x[1])
print(f'\n→ Best method: {best_name} (OOF: {best_r:.4f})')

# Ranking diagnostics
sp, _ = spearmanr(y, final_oof)
print(f'  Spearman rank correlation: {sp:.4f}')

# --- 4. Within-season calibration (test predictions only) ---
# Uses known training seeds per season to calibrate the model output
cal_pred = np.copy(final_pred)
for season in sorted(train_tourn['Season'].unique()):
    tr_m = (train_tourn['Season'].values == season)
    te_m = (test_tourn['Season'].values == season)
    tp = np.where(tr_m)[0]; ep = np.where(te_m)[0]
    iso = IsotonicRegression(y_min=1, y_max=68, increasing=True, out_of_bounds='clip')
    iso.fit(final_oof[tp], y[tp])
    cal_pred[ep] = np.clip(iso.predict(final_pred[ep]), 1, 68)
print('Within-season calibrated test predictions ready')

Optimised Blend OOF: 3.9213  (176,851 combos searched)
  Ridge: 23%
  XGB: 62%
  HGB: 15%
Stacking OOF: 4.1142
Average OOF:  4.0037

→ Best method: blend (OOF: 3.9213)
  Spearman rank correlation: 0.9784
Within-season calibrated test predictions ready


In [16]:
# === HUNGARIAN ASSIGNMENT + SUBMISSIONS ===
import glob

non_tourn_ids = test_data[test_data['Bid Type'].isna()]['RecordID'].values

def hungarian_sub(preds, fname):
    """Apply Hungarian assignment per season and create submission CSV."""
    s = sub_template.copy()
    pm = {r: 0 for r in non_tourn_ids}
    for season in sorted(test_data['Season'].unique()):
        ts = dataset_df[(dataset_df['Season'] == season) &
                        (dataset_df['Overall Seed'].notna())]
        known = set(ts['Overall Seed'].astype(int).values)
        available = sorted(set(range(1, 69)) - known)
        te = test_tourn[test_tourn['Season'] == season]
        pos = [np.where(test_tourn.index == i)[0][0] for i in te.index]
        ps = preds[pos]
        rids = te['RecordID'].values
        cost = np.array([[(p - a)**2 for a in available] for p in ps])
        ri, ci = linear_sum_assignment(cost)
        for i, j in zip(ri, ci):
            pm[rids[i]] = available[j]
    s['Overall Seed'] = s['RecordID'].map(pm)
    s.to_csv(fname, index=False)
    nz = (s['Overall Seed'] > 0.5).sum()
    print(f'  {fname:30s} nz={nz}  mean_seed={s["Overall Seed"][s["Overall Seed"]>0].mean():.1f}')
    return fname

print('=== Generating Submissions (Hungarian Assignment) ===\n')
pred_rank = test_tourn['tourn_net_rank'].values.astype(float)

# Core variants
hungarian_sub(final_pred,    'sub_best.csv')
hungarian_sub(blend_pred,    'sub_blend.csv')
hungarian_sub(stk_pred,      'sub_stack.csv')
hungarian_sub(cal_pred,      'sub_cal.csv')

# Rank mixtures (model + NET rank anchor)
for a in [0.10, 0.15, 0.20, 0.30, 0.50]:
    hungarian_sub((1-a)*final_pred + a*pred_rank, f'sub_mix{int(a*100)}.csv')

# Within-season models
hungarian_sub(ws_pred['WS_Iso'],   'sub_ws_iso.csv')
hungarian_sub(ws_pred['Cal_XGB'],  'sub_cal_xgb.csv')
hungarian_sub(ws_pred['Cal_Blend'],'sub_cal_blend.csv')

# Pure rank baseline
hungarian_sub(pred_rank, 'sub_rank_only.csv')

# Save all to submissions/
dest = 'submissions'
os.makedirs(dest, exist_ok=True)
for f in glob.glob('sub_*.csv'):
    shutil.copy(f, dest)

print(f'\nAll saved to {dest}/')
print('\nRecommended submission order:')
print('  1. sub_cal.csv          (within-season calibrated best)')
print('  2. sub_best.csv         (best ensemble)')
print('  3. sub_mix15.csv        (85% model + 15% rank)')
print('  4. sub_blend.csv        (optimised blend)')
print('  5. sub_cal_xgb.csv      (calibrated XGB)')
print('  6. sub_cal_blend.csv    (calibrated blend)')

=== Generating Submissions (Hungarian Assignment) ===

  sub_best.csv                   nz=91  mean_seed=37.6
  sub_blend.csv                  nz=91  mean_seed=37.6
  sub_stack.csv                  nz=91  mean_seed=37.6
  sub_cal.csv                    nz=91  mean_seed=37.6
  sub_mix10.csv                  nz=91  mean_seed=37.6
  sub_mix15.csv                  nz=91  mean_seed=37.6
  sub_mix20.csv                  nz=91  mean_seed=37.6
  sub_mix30.csv                  nz=91  mean_seed=37.6
  sub_mix50.csv                  nz=91  mean_seed=37.6
  sub_ws_iso.csv                 nz=91  mean_seed=37.6
  sub_cal_xgb.csv                nz=91  mean_seed=37.6
  sub_cal_blend.csv              nz=91  mean_seed=37.6
  sub_rank_only.csv              nz=91  mean_seed=37.6

All saved to submissions/

Recommended submission order:
  1. sub_cal.csv          (within-season calibrated best)
  2. sub_best.csv         (best ensemble)
  3. sub_mix15.csv        (85% model + 15% rank)
  4. sub_blend.csv     

In [17]:
# === ENHANCED MODELS: push OOF lower ===
print('=== Additional Models ===')
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)

# 1. XGB with early stopping (lower LR → better generalisation)
oof_es = np.zeros(len(X)); pred_es = np.zeros(len(Xt))
for tri, vai in kf5.split(X):
    sc = RobustScaler()
    Xtr, Xva, Xte = sc.fit_transform(X[tri]), sc.transform(X[vai]), sc.transform(Xt)
    m = xgb.XGBRegressor(n_estimators=5000, max_depth=4, learning_rate=0.01,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=2.0,
        min_child_weight=3, tree_method='hist', random_state=42, n_jobs=-1,
        early_stopping_rounds=100)
    m.fit(Xtr, y[tri], eval_set=[(Xva, y[vai])], verbose=False)
    oof_es[vai] = np.clip(m.predict(Xva), 1, 68)
    pred_es += np.clip(m.predict(Xte), 1, 68) / 5
print(f'  XGB-ES    OOF: {np.sqrt(mean_squared_error(y, oof_es)):.4f}')

# 2. Deeper XGB (captures more interactions)
oof_d6 = np.zeros(len(X)); pred_d6 = np.zeros(len(Xt))
for tri, vai in kf5.split(X):
    sc = RobustScaler()
    Xtr, Xva, Xte = sc.fit_transform(X[tri]), sc.transform(X[vai]), sc.transform(Xt)
    m = xgb.XGBRegressor(n_estimators=600, max_depth=6, learning_rate=0.03,
        subsample=0.7, colsample_bytree=0.6, reg_alpha=1.0, reg_lambda=3.0,
        min_child_weight=5, tree_method='hist', random_state=99, n_jobs=-1)
    m.fit(Xtr, y[tri], verbose=False)
    oof_d6[vai] = np.clip(m.predict(Xva), 1, 68)
    pred_d6 += np.clip(m.predict(Xte), 1, 68) / 5
print(f'  XGB-D6    OOF: {np.sqrt(mean_squared_error(y, oof_d6)):.4f}')

# 3. ExtraTrees (tree-bag diversity)
from sklearn.ensemble import ExtraTreesRegressor
oof_et = np.zeros(len(X)); pred_et = np.zeros(len(Xt))
for tri, vai in kf5.split(X):
    sc = RobustScaler()
    Xtr, Xva, Xte = sc.fit_transform(X[tri]), sc.transform(X[vai]), sc.transform(Xt)
    m = ExtraTreesRegressor(n_estimators=500, max_depth=12, min_samples_leaf=3,
        max_features=0.7, random_state=42, n_jobs=-1)
    m.fit(Xtr, y[tri])
    oof_et[vai] = np.clip(m.predict(Xva), 1, 68)
    pred_et += np.clip(m.predict(Xte), 1, 68) / 5
print(f'  ExtraT    OOF: {np.sqrt(mean_squared_error(y, oof_et)):.4f}')

# --- Enhanced 7-model blend (grid + Nelder-Mead refine) ---
print('\n=== Enhanced 7-Model Blend ===')
all7_names = ['Ridge', 'XGB', 'LGB', 'HGB', 'XGB-ES', 'XGB-D6', 'ExtraT']
all7_oof  = np.column_stack([cs_oof_ridge, cs_oof_xgb, cs_oof_lgb, cs_oof_hgb,
                              oof_es, oof_d6, oof_et])
all7_pred = np.column_stack([cs_pred_ridge, cs_pred_xgb, cs_pred_lgb, cs_pred_hgb,
                              pred_es, pred_d6, pred_et])

def loss7(w):
    w = np.abs(w) / np.abs(w).sum()
    return np.sqrt(mean_squared_error(y, np.clip(all7_oof @ w, 1, 68)))

best7_r, best7_w = 999, None
for seed in range(500):
    x0 = np.random.RandomState(seed).dirichlet(np.ones(7))
    res = minimize(loss7, x0=x0, method='Nelder-Mead', options={'maxiter': 3000})
    ww = np.abs(res.x) / np.abs(res.x).sum()
    if res.fun < best7_r:
        best7_r, best7_w = res.fun, ww

blend7_oof  = np.clip(all7_oof  @ best7_w, 1, 68)
blend7_pred = np.clip(all7_pred @ best7_w, 1, 68)
print(f'Enhanced Blend OOF: {best7_r:.4f}')
for nm, w in zip(all7_names, best7_w):
    if w > 0.005: print(f'  {nm}: {w:.1%}')

# Stacking on 7 models
X_stk7    = np.column_stack([all7_oof,  X[:,  [features.index(f) for f in
              ['tourn_net_rank','NET','quality_wins','seed_line_est']]]])
X_stk7_te = np.column_stack([all7_pred, Xt[:, [features.index(f) for f in
              ['tourn_net_rank','NET','quality_wins','seed_line_est']]]])
oof_stk7 = np.zeros(len(X))
for tri, vai in kf5.split(X_stk7):
    sc = RobustScaler()
    oof_stk7[vai] = np.clip(
        Ridge(alpha=5).fit(sc.fit_transform(X_stk7[tri]), y[tri])
        .predict(sc.transform(X_stk7[vai])), 1, 68)
stk7_rmse = np.sqrt(mean_squared_error(y, oof_stk7))
sc_f7 = RobustScaler()
stk7_pred = np.clip(
    Ridge(alpha=5).fit(sc_f7.fit_transform(X_stk7), y)
    .predict(sc_f7.transform(X_stk7_te)), 1, 68)
print(f'Stacking-7 OOF:    {stk7_rmse:.4f}')

# Average
avg7_oof  = 0.5*blend7_oof + 0.5*oof_stk7
avg7_pred = 0.5*blend7_pred + 0.5*stk7_pred
avg7_rmse = np.sqrt(mean_squared_error(y, avg7_oof))
print(f'Average-7 OOF:     {avg7_rmse:.4f}')

choices7 = [('blend7', best7_r, blend7_pred, blend7_oof),
            ('stack7', stk7_rmse, stk7_pred, oof_stk7),
            ('avg7',   avg7_rmse, avg7_pred, avg7_oof)]
best7_name, best7_final_r, best7_final_pred, best7_final_oof = \
    min(choices7, key=lambda x: x[1])
print(f'\n→ Best enhanced: {best7_name} (OOF: {best7_final_r:.4f})')
print(f'  vs original:   {best_name} (OOF: {best_r:.4f})')

use_v2 = best7_final_r < best_r
if use_v2:
    print('  ✓ Enhanced is BETTER — using it')
    best_all_pred, best_all_oof = best7_final_pred, best7_final_oof
else:
    print('  Original is still best — keeping it')
    best_all_pred, best_all_oof = final_pred, final_oof

# Within-season calibration
cal_all = np.copy(best_all_pred)
for season in sorted(train_tourn['Season'].unique()):
    tr_m = (train_tourn['Season'].values == season)
    te_m = (test_tourn['Season'].values == season)
    tp = np.where(tr_m)[0]; ep = np.where(te_m)[0]
    iso = IsotonicRegression(y_min=1, y_max=68, increasing=True, out_of_bounds='clip')
    iso.fit(best_all_oof[tp], y[tp])
    cal_all[ep] = np.clip(iso.predict(best_all_pred[ep]), 1, 68)

# --- New submissions ---
print('\n=== Enhanced Submissions ===')
hungarian_sub(best_all_pred, 'sub_v2_best.csv')
hungarian_sub(cal_all,       'sub_v2_cal.csv')
for a in [0.10, 0.15, 0.20]:
    hungarian_sub((1-a)*best_all_pred + a*pred_rank, f'sub_v2_mix{int(a*100)}.csv')

for f in glob.glob('sub_v2_*.csv'):
    shutil.copy(f, dest)
print(f'\nSaved to {dest}/')

=== Additional Models ===
  XGB-ES    OOF: 4.2085
  XGB-D6    OOF: 4.1910
  ExtraT    OOF: 4.8759

=== Enhanced 7-Model Blend ===
Enhanced Blend OOF: 3.9214
  Ridge: 22.7%
  XGB: 63.8%
  HGB: 13.4%
Stacking-7 OOF:    4.2075
Average-7 OOF:     4.0472

→ Best enhanced: blend7 (OOF: 3.9214)
  vs original:   blend (OOF: 3.9213)
  Original is still best — keeping it

=== Enhanced Submissions ===
  sub_v2_best.csv                nz=91  mean_seed=37.6
  sub_v2_cal.csv                 nz=91  mean_seed=37.6
  sub_v2_mix10.csv               nz=91  mean_seed=37.6
  sub_v2_mix15.csv               nz=91  mean_seed=37.6
  sub_v2_mix20.csv               nz=91  mean_seed=37.6

Saved to submissions/


In [18]:
# === PAIRWISE RANKING MODEL ===
# Fundamentally different: predict "is team A better than B?" for all pairs
# Gives ~6000 training pairs vs 249 team samples → more data, direct ranking optimisation
from itertools import combinations

seasons = sorted(train_tourn['Season'].unique())

# 1. Build pairwise training data (within-season pairs)
pair_X, pair_y, pair_season = [], [], []
for season in seasons:
    idx = np.where(train_tourn['Season'].values == season)[0]
    for i, j in combinations(idx, 2):
        pair_X.append(X[i] - X[j])
        pair_y.append(1.0 if y[i] < y[j] else 0.0)
        pair_season.append(season)
pair_X = np.array(pair_X, dtype=np.float32)
pair_y = np.array(pair_y)
pair_season = np.array(pair_season)
print(f'Pairwise pairs: {len(pair_X)}')

# 2. Train global pairwise classifier
pw_global = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7, random_state=42, n_jobs=-1,
    eval_metric='logloss')
pw_global.fit(pair_X, pair_y, verbose=False)

# Quick accuracy check
from sklearn.model_selection import cross_val_score
acc = cross_val_score(pw_global, pair_X, pair_y, cv=5, scoring='accuracy')
print(f'Pairwise accuracy: {acc.mean():.4f} ± {acc.std():.4f}')

# 3. Convert pairwise predictions to seeds  
# For each team: compare against all same-season training teams  
# fraction beaten → seed via per-season isotonic calibration

# Helper: compute "fraction beaten" for a set of teams vs reference teams
def get_fractions(target_X, ref_X, model):
    fracs = np.zeros(len(target_X))
    for i in range(len(target_X)):
        diffs = target_X[i] - ref_X
        probs = model.predict_proba(diffs)[:, 1]
        fracs[i] = probs.mean()
    return fracs

# 4. Test predictions (global model + per-season calibration)
pw_test = np.zeros(len(Xt))
for season in seasons:
    tr_idx = np.where(train_tourn['Season'].values == season)[0]
    te_idx = np.where(test_tourn['Season'].values == season)[0]
    
    # Calibrate using training teams: fraction → seed
    tr_fracs = get_fractions(X[tr_idx], X[tr_idx], pw_global)
    iso = IsotonicRegression(y_min=1, y_max=68, increasing=False, out_of_bounds='clip')
    iso.fit(tr_fracs, y[tr_idx])  # Higher fraction beaten → lower seed
    
    te_fracs = get_fractions(Xt[te_idx], X[tr_idx], pw_global)
    pw_test[te_idx] = np.clip(iso.predict(te_fracs), 1, 68)

# 5. Leave-one-season-out OOF (proper: retrain pairwise model on other seasons)
pw_oof = np.zeros(len(X))
for val_season in seasons:
    # Build pairs from other seasons only
    lp_mask = (pair_season != val_season)
    pw_cv = xgb.XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, random_state=42, n_jobs=-1,
        eval_metric='logloss')
    pw_cv.fit(pair_X[lp_mask], pair_y[lp_mask], verbose=False)
    
    val_idx = np.where(train_tourn['Season'].values == val_season)[0]
    # LOO within validation season: compare each team against the rest
    for i_local in range(len(val_idx)):
        i_global = val_idx[i_local]
        others = np.delete(val_idx, i_local)
        diffs = X[i_global] - X[others]
        probs = pw_cv.predict_proba(diffs)[:, 1]
        frac = probs.mean()
        # Linear mapping as fallback (calibration needs full data)
        pw_oof[i_global] = np.clip(1 + (1 - frac) * 67, 1, 68)

pw_rmse = np.sqrt(mean_squared_error(y, pw_oof))
sp_pw, _ = spearmanr(y, pw_oof)
print(f'Pairwise OOF RMSE: {pw_rmse:.4f}, Spearman: {sp_pw:.4f}')

# 6. Blend pairwise with regression models
all_pw = np.column_stack([cs_oof_ridge, cs_oof_xgb, cs_oof_lgb, cs_oof_hgb, pw_oof])
tst_pw = np.column_stack([cs_pred_ridge, cs_pred_xgb, cs_pred_lgb, cs_pred_hgb, pw_test])
nm_pw = ['Ridge', 'XGB', 'LGB', 'HGB', 'Pairwise']

best_pw_r, best_pw_w = 999, None
for seed in range(500):
    x0 = np.random.RandomState(seed).dirichlet(np.ones(5))
    res = minimize(lambda w: np.sqrt(mean_squared_error(y,
        np.clip(all_pw @ (np.abs(w)/np.abs(w).sum()), 1, 68))),
        x0=x0, method='Nelder-Mead', options={'maxiter': 3000})
    w = np.abs(res.x) / np.abs(res.x).sum()
    if res.fun < best_pw_r: best_pw_r, best_pw_w = res.fun, w

pw_blend = np.clip(tst_pw @ best_pw_w, 1, 68)
pw_blend_oof = np.clip(all_pw @ best_pw_w, 1, 68)
print(f'\nBlend+PW OOF: {best_pw_r:.4f}')
for nm, w in zip(nm_pw, best_pw_w):
    if w > 0.005: print(f'  {nm}: {w:.1%}')

# Compare
print(f'\n  Original blend OOF:   {best_rmse:.4f}')
print(f'  With pairwise blend:  {best_pw_r:.4f}')
improve = best_pw_r < best_rmse
print(f'  → {"IMPROVED!" if improve else "No improvement"}')

# Submissions
print('\n=== Pairwise Submissions ===')
hungarian_sub(pw_blend, 'sub_pw_blend.csv')
hungarian_sub(pw_test, 'sub_pw_only.csv')

# Calibrated pairwise blend
cal_pw = np.copy(pw_blend)
for season in seasons:
    tr_m = (train_tourn['Season'].values == season)
    te_m = (test_tourn['Season'].values == season)
    tp = np.where(tr_m)[0]; ep = np.where(te_m)[0]
    iso = IsotonicRegression(y_min=1, y_max=68, increasing=True, out_of_bounds='clip')
    iso.fit(pw_blend_oof[tp], y[tp])
    cal_pw[ep] = np.clip(iso.predict(pw_blend[ep]), 1, 68)
hungarian_sub(cal_pw, 'sub_pw_cal.csv')

for f in glob.glob('sub_pw_*.csv'):
    shutil.copy(f, dest)
print(f'Saved to {dest}/')

Pairwise pairs: 6093
Pairwise accuracy: 0.9522 ± 0.0151
Pairwise OOF RMSE: 3.8543, Spearman: 0.9822

Blend+PW OOF: 3.5479
  Ridge: 13.2%
  XGB: 25.9%
  HGB: 8.2%
  Pairwise: 52.7%

  Original blend OOF:   3.9213
  With pairwise blend:  3.5479
  → IMPROVED!

=== Pairwise Submissions ===
  sub_pw_blend.csv               nz=91  mean_seed=37.6
  sub_pw_only.csv                nz=91  mean_seed=37.6
  sub_pw_cal.csv                 nz=91  mean_seed=37.6
Saved to submissions/


In [19]:
# === IMPROVED PAIRWISE: bidirectional pairs + isotonic calibrated OOF ===
print('=== Improved Pairwise (v3) ===')

# 1. Bidirectional pairs (A-B and B-A: symmetric learning, 2x data)
bi_X, bi_y, bi_s = [], [], []
for season in seasons:
    idx = np.where(train_tourn['Season'].values == season)[0]
    for i, j in combinations(idx, 2):
        d = X[i] - X[j]
        label = 1.0 if y[i] < y[j] else 0.0
        bi_X.append(d);  bi_y.append(label);     bi_s.append(season)
        bi_X.append(-d); bi_y.append(1 - label); bi_s.append(season)
bi_X = np.array(bi_X, dtype=np.float32)
bi_y = np.array(bi_y)
bi_s = np.array(bi_s)
print(f'Bidirectional pairs: {len(bi_X)}')

pw3 = xgb.XGBClassifier(n_estimators=500, max_depth=5, learning_rate=0.03,
    subsample=0.8, colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=2.0,
    random_state=42, n_jobs=-1, eval_metric='logloss')
pw3.fit(bi_X, bi_y, verbose=False)

# 2. Vectorised Copeland scores + per-season isotonic calibration
def copeland_scores(target_X, ref_X, model):
    """Copeland score = total expected wins against reference teams."""
    n_t, n_r = len(target_X), len(ref_X)
    all_diffs = np.vstack([target_X[i] - ref_X for i in range(n_t)])  # (n_t*n_r, F)
    all_probs = model.predict_proba(all_diffs)[:, 1]
    return all_probs.reshape(n_t, n_r).sum(axis=1)  # (n_t,)

# Test predictions
pw3_test = np.zeros(len(Xt))
for season in seasons:
    tr_idx = np.where(train_tourn['Season'].values == season)[0]
    te_idx = np.where(test_tourn['Season'].values == season)[0]
    # Training Copeland (for calibration), LOO: exclude self
    n_tr_s = len(tr_idx)
    tr_cops = np.zeros(n_tr_s)
    for i in range(n_tr_s):
        others = np.delete(tr_idx, i)
        diffs = X[tr_idx[i]] - X[others]
        tr_cops[i] = pw3.predict_proba(diffs)[:, 1].sum()
    te_cops = copeland_scores(Xt[te_idx], X[tr_idx], pw3)
    # Isotonic calibration: Copeland → seed (more wins = lower seed)
    iso = IsotonicRegression(y_min=1, y_max=68, increasing=False, out_of_bounds='clip')
    iso.fit(tr_cops, y[tr_idx])
    pw3_test[te_idx] = np.clip(iso.predict(te_cops), 1, 68)

# 3. LOO-season OOF with per-season LOO isotonic calibration
pw3_oof = np.zeros(len(X))
for val_season in seasons:
    lp_mask = (bi_s != val_season)
    pw_cv3 = xgb.XGBClassifier(n_estimators=500, max_depth=5, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.7, reg_alpha=0.5, reg_lambda=2.0,
        random_state=42, n_jobs=-1, eval_metric='logloss')
    pw_cv3.fit(bi_X[lp_mask], bi_y[lp_mask], verbose=False)
    
    val_idx = np.where(train_tourn['Season'].values == val_season)[0]
    n_val = len(val_idx)
    
    # Copeland scores for all val teams
    val_cops = np.zeros(n_val)
    for i in range(n_val):
        others = np.delete(val_idx, i)
        diffs = X[val_idx[i]] - X[others]
        val_cops[i] = pw_cv3.predict_proba(diffs)[:, 1].sum()
    
    # LOO isotonic calibration within val season
    for i in range(n_val):
        mask = np.ones(n_val, bool); mask[i] = False
        iso = IsotonicRegression(y_min=1, y_max=68, increasing=False, out_of_bounds='clip')
        iso.fit(val_cops[mask], y[val_idx[mask]])
        pw3_oof[val_idx[i]] = np.clip(iso.predict([val_cops[i]])[0], 1, 68)

pw3_rmse = np.sqrt(mean_squared_error(y, pw3_oof))
sp3, _ = spearmanr(y, pw3_oof)
print(f'PW3 OOF RMSE: {pw3_rmse:.4f}, Spearman: {sp3:.4f}  (PW1: {pw_rmse:.4f})')

# 4. Blend with regression models
all_pw3 = np.column_stack([cs_oof_ridge, cs_oof_xgb, cs_oof_lgb, cs_oof_hgb, pw3_oof])
tst_pw3 = np.column_stack([cs_pred_ridge, cs_pred_xgb, cs_pred_lgb, cs_pred_hgb, pw3_test])

best_pw3_r, best_pw3_w = 999, None
for seed in range(500):
    x0 = np.random.RandomState(seed).dirichlet(np.ones(5))
    res = minimize(lambda w: np.sqrt(mean_squared_error(y,
        np.clip(all_pw3 @ (np.abs(w)/np.abs(w).sum()), 1, 68))),
        x0=x0, method='Nelder-Mead', options={'maxiter': 3000})
    w = np.abs(res.x) / np.abs(res.x).sum()
    if res.fun < best_pw3_r: best_pw3_r, best_pw3_w = res.fun, w

pw3_blend = np.clip(tst_pw3 @ best_pw3_w, 1, 68)
pw3_blend_oof = np.clip(all_pw3 @ best_pw3_w, 1, 68)
print(f'\nPW3 Blend OOF: {best_pw3_r:.4f}  (PW1 blend: {best_pw_r:.4f})')
for nm, w in zip(nm_pw, best_pw3_w):
    if w > 0.005: print(f'  {nm}: {w:.1%}')

# 5. Final submissions
print('\n=== v3 Submissions ===')
hungarian_sub(pw3_blend, 'sub_pw3_blend.csv')
hungarian_sub(pw3_test,  'sub_pw3_only.csv')

# Calibrated
cal_pw3 = np.copy(pw3_blend)
for season in seasons:
    tr_m = (train_tourn['Season'].values == season)
    te_m = (test_tourn['Season'].values == season)
    tp = np.where(tr_m)[0]; ep = np.where(te_m)[0]
    iso = IsotonicRegression(y_min=1, y_max=68, increasing=True, out_of_bounds='clip')
    iso.fit(pw3_blend_oof[tp], y[tp])
    cal_pw3[ep] = np.clip(iso.predict(pw3_blend[ep]), 1, 68)
hungarian_sub(cal_pw3, 'sub_pw3_cal.csv')

# Rank mixtures
for a in [0.10, 0.15]:
    hungarian_sub((1-a)*pw3_blend + a*pred_rank, f'sub_pw3_mix{int(a*100)}.csv')

for f in glob.glob('sub_pw3_*.csv'):
    shutil.copy(f, dest)
print(f'Saved to {dest}/')

=== Improved Pairwise (v3) ===
Bidirectional pairs: 12186
PW3 OOF RMSE: 3.8156, Spearman: 0.9781  (PW1: 3.8543)

PW3 Blend OOF: 3.5924  (PW1 blend: 3.5479)
  Ridge: 6.6%
  XGB: 33.6%
  HGB: 4.5%
  Pairwise: 55.3%

=== v3 Submissions ===
  sub_pw3_blend.csv              nz=91  mean_seed=37.6
  sub_pw3_only.csv               nz=91  mean_seed=37.6
  sub_pw3_cal.csv                nz=91  mean_seed=37.6
  sub_pw3_mix10.csv              nz=91  mean_seed=37.6
  sub_pw3_mix15.csv              nz=91  mean_seed=37.6
Saved to submissions/


In [20]:
# === FINAL: Combine ALL models for ultimate prediction ===
print('=== Ultimate 6-Model Blend (4 regression + 2 pairwise) ===')

all6 = np.column_stack([cs_oof_ridge, cs_oof_xgb, cs_oof_lgb, cs_oof_hgb, pw_oof, pw3_oof])
tst6 = np.column_stack([cs_pred_ridge, cs_pred_xgb, cs_pred_lgb, cs_pred_hgb, pw_test, pw3_test])
nm6 = ['Ridge', 'XGB', 'LGB', 'HGB', 'PW1', 'PW3']

best6_r, best6_w = 999, None
for seed in range(500):
    x0 = np.random.RandomState(seed).dirichlet(np.ones(6))
    res = minimize(lambda w: np.sqrt(mean_squared_error(y,
        np.clip(all6 @ (np.abs(w)/np.abs(w).sum()), 1, 68))),
        x0=x0, method='Nelder-Mead', options={'maxiter': 3000})
    w = np.abs(res.x) / np.abs(res.x).sum()
    if res.fun < best6_r: best6_r, best6_w = res.fun, w

blend6 = np.clip(tst6 @ best6_w, 1, 68)
blend6_oof = np.clip(all6 @ best6_w, 1, 68)
print(f'6-Model Blend OOF: {best6_r:.4f}')
for nm, w in zip(nm6, best6_w):
    if w > 0.005: print(f'  {nm}: {w:.1%}')

# Stacking on all 6 models + key features
top_fi = [features.index(f) for f in ['tourn_net_rank', 'NET', 'quality_wins', 'seed_line_est']]
X_stk6 = np.column_stack([all6, X[:, top_fi]])
X_stk6_te = np.column_stack([tst6, Xt[:, top_fi]])

kf = KFold(n_splits=5, shuffle=True, random_state=99)
oof_stk6 = np.zeros(len(X))
for tri, vai in kf.split(X_stk6):
    sc = RobustScaler()
    oof_stk6[vai] = np.clip(
        Ridge(alpha=5).fit(sc.fit_transform(X_stk6[tri]), y[tri])
        .predict(sc.transform(X_stk6[vai])), 1, 68)
stk6_rmse = np.sqrt(mean_squared_error(y, oof_stk6))
sc_f = RobustScaler()
stk6_pred = np.clip(
    Ridge(alpha=5).fit(sc_f.fit_transform(X_stk6), y)
    .predict(sc_f.transform(X_stk6_te)), 1, 68)
print(f'Stacking-6 OOF:    {stk6_rmse:.4f}')

# Average blend + stacking
avg6_oof = 0.5 * blend6_oof + 0.5 * oof_stk6
avg6_pred = 0.5 * blend6 + 0.5 * stk6_pred
avg6_rmse = np.sqrt(mean_squared_error(y, avg6_oof))
print(f'Average-6 OOF:     {avg6_rmse:.4f}')

# Pick best overall
all_methods = [
    ('pw1_blend',  best_pw_r,   np.clip(tst_pw @ best_pw_w, 1, 68),   np.clip(all_pw @ best_pw_w, 1, 68)),
    ('pw3_blend',  best_pw3_r,  pw3_blend,  pw3_blend_oof),
    ('6model_blend', best6_r,   blend6,     blend6_oof),
    ('6model_stack', stk6_rmse, stk6_pred,  oof_stk6),
    ('6model_avg',   avg6_rmse, avg6_pred,  avg6_oof),
]
best_overall = min(all_methods, key=lambda x: x[1])
print(f'\n→ BEST OVERALL: {best_overall[0]} (OOF: {best_overall[1]:.4f})')
best_final_pred = best_overall[2]
best_final_oof = best_overall[3]

# Within-season calibration on the absolute best
cal_final = np.copy(best_final_pred)
for season in seasons:
    tr_m = (train_tourn['Season'].values == season)
    te_m = (test_tourn['Season'].values == season)
    tp = np.where(tr_m)[0]; ep = np.where(te_m)[0]
    iso = IsotonicRegression(y_min=1, y_max=68, increasing=True, out_of_bounds='clip')
    iso.fit(best_final_oof[tp], y[tp])
    cal_final[ep] = np.clip(iso.predict(best_final_pred[ep]), 1, 68)

# FINAL submissions
print('\n=== FINAL Submissions ===')
hungarian_sub(best_final_pred, 'sub_FINAL_best.csv')
hungarian_sub(cal_final,       'sub_FINAL_cal.csv')
for a in [0.10, 0.15, 0.20]:
    hungarian_sub((1-a)*best_final_pred + a*pred_rank, f'sub_FINAL_mix{int(a*100)}.csv')

for f in glob.glob('sub_FINAL_*.csv'):
    shutil.copy(f, dest)

# Summary
print(f'\n{"="*50}')
print(f'SCORE PROGRESSION:')
print(f'  Regression-only blend:    OOF = 3.9213')
print(f'  + Pairwise PW1 blend:     OOF = {best_pw_r:.4f}  ({(1-best_pw_r/3.9213)*100:.1f}% better)')
print(f'  + Pairwise PW3 blend:     OOF = {best_pw3_r:.4f}')
print(f'  6-model blend:            OOF = {best6_r:.4f}')
print(f'  BEST OVERALL:             OOF = {best_overall[1]:.4f}')
print(f'\nRecommended submission order:')
print(f'  1. sub_FINAL_best.csv      (best overall method)')
print(f'  2. sub_FINAL_cal.csv       (within-season calibrated)')
print(f'  3. sub_pw_blend.csv        (pairwise PW1 blend)')
print(f'  4. sub_FINAL_mix15.csv     (85% model + 15% rank)')
print(f'  5. sub_pw_cal.csv          (PW1 calibrated)')
print(f'{"="*50}')

=== Ultimate 6-Model Blend (4 regression + 2 pairwise) ===
6-Model Blend OOF: 3.4900
  Ridge: 7.7%
  XGB: 22.4%
  HGB: 4.9%
  PW1: 35.4%
  PW3: 29.6%
Stacking-6 OOF:    3.6873
Average-6 OOF:     3.5521

→ BEST OVERALL: 6model_blend (OOF: 3.4900)

=== FINAL Submissions ===
  sub_FINAL_best.csv             nz=91  mean_seed=37.6
  sub_FINAL_cal.csv              nz=91  mean_seed=37.6
  sub_FINAL_mix10.csv            nz=91  mean_seed=37.6
  sub_FINAL_mix15.csv            nz=91  mean_seed=37.6
  sub_FINAL_mix20.csv            nz=91  mean_seed=37.6

SCORE PROGRESSION:
  Regression-only blend:    OOF = 3.9213
  + Pairwise PW1 blend:     OOF = 3.5479  (9.5% better)
  + Pairwise PW3 blend:     OOF = 3.5924
  6-model blend:            OOF = 3.4900
  BEST OVERALL:             OOF = 3.4900

Recommended submission order:
  1. sub_FINAL_best.csv      (best overall method)
  2. sub_FINAL_cal.csv       (within-season calibrated)
  3. sub_pw_blend.csv        (pairwise PW1 blend)
  4. sub_FINAL_mix15.csv